In [1]:
#Importing Necessary Packages
import pandas as pd
import os

from importlib.metadata import version, PackageNotFoundError

In [2]:
packages = ["pandas"]

for p in packages:
    try:
        print(f"{p}=={version(p)}")
    except PackageNotFoundError:
        print(f"{p} Not Found")

pandas==2.3.0


In [3]:
#Directory from where we will find our participant decisions
dir2 = "../DecisionData/converted/"
#Four group conditions
conds = ['CM1', 'CM2', 'IM1', 'IM2']
#8 Moral Decisions used to split the dataframe
decs = ['CHOICE 1 - Sausage','CHOICE 2 -  Steal from Mick','CHOICE 3 - Kick the dog','CHOICE 4 - Coin toss','CHOICE 5 - Steal from Andy','CHOICE 6 - Pull the lever','CHOICE 7 - Kick the chair','CHOICE 8 - Shoot Harry']
count = 0

#To Separate each Question to its own csv file 
for c in conds:
    count = 0
    df_cond = pd.DataFrame()
    df_temp = pd.DataFrame()
    df_list = []

    for main in os.listdir(dir2):
        if main == c:
            new = os.path.join(dir2,main)
            for folder in os.listdir(new):
                final = os.path.join(new,folder)
                for file in os.listdir(final):
                    df_temp = pd.read_csv(os.path.join(final,file))
                    df_list.append(df_temp)
            break

    #To take all of the decisions of a specific condition into one csv file 
    df_cond = pd.concat(df_list, ignore_index=True)
    df_cond.to_csv(f"../Csv Files/Conditions_Full/{c}_Decisions.csv", index = False)
    #To go through each moral decision and put them into their own csv file
    for d in decs:
        count += 1
        df_Q = df_cond[df_cond['passage'] == d]
        df_Q.to_csv(f"../Csv Files/Conditions_Separate/{c}/Q{count}_{c}.csv", index = False)

In [4]:
#Combining the four full condition csv files created earlier into one full file

CM1 = pd.read_csv("../Csv Files/Conditions_Full/CM1_Decisions.csv")
CM2 = pd.read_csv("../Csv Files/Conditions_Full/CM2_Decisions.csv")
IM1 = pd.read_csv("../Csv Files/Conditions_Full/IM1_Decisions.csv")
IM2 = pd.read_csv("../Csv Files/Conditions_Full/IM2_Decisions.csv")

df_final = pd.concat([CM1,CM2,IM1,IM2], ignore_index=True)
df_final.to_csv("../Csv Files/Final_Full.csv", index = False)

In [5]:
#Removing participants from the Eye-Tracking Data that had a 10 second timer for their decisions

dir = "../Not Needed/Experiments (10 Second Timer)"
dir2 = "../Eye Data/Submission2_Data.tsv"
remove = []

for f in os.listdir(dir):
    remove.append(f)

df = pd.read_csv(dir2, sep="\t")

df = df[~df["Recording"].isin(remove)]
df = df.reset_index(drop=True)

df.rename(columns={"Participant":"testerKey"}, inplace=True)
df.to_csv("../Eye Data/Final_Cleaned_Data.csv", index = False)

In [6]:
#Taking all the data gathered and cleaned during the experiments and merging into one file
df_eye = pd.read_csv("../Eye Data/Final_Cleaned_Data.csv")
df_mac = pd.read_csv("../Csv Files/MAC.csv")
df_post = pd.read_csv("../Csv Files/Post.csv")
df_decisions = pd.read_csv("../Csv Files/Final_Full.csv")

df_merged = df_eye.merge(df_mac, on="testerKey", how="inner") \
               .merge(df_post, on="testerKey", how="left") \
               .merge(df_decisions, on="testerKey", how="inner")

#This file will be amended to match our requirements for the project
df_merged.to_csv("../Csv Files/merged.csv",index=False)

In [7]:
df = pd.read_csv("../Csv Files/merged.csv")
#Adding a column that includes simply the number of the decision to match with the eye-tracking data rows
df["passage_num"] = df["passage"].str.extract(r'(\d+)').astype(int)
keys = df["testerKey"].unique()
interval = (6,7,8)

#Creating two different dataframes

#Contains the Entire Recording of the morality meter gaze data
data_full_recording = df[(df["TOI"] == "Entire Recording") & (df["AOI"] == "Morality Meter") & (df["passage_num"] == 6)]

#Taking the data concerning our required passages (6,7,8)
data = df[((df["testerKey"].isin(keys)) & (df["Interval"].isin(interval)) & (df["Interval"] == df["passage_num"]))].copy()
data.reset_index(drop=True, inplace=True)
data_full_recording.reset_index(drop=True, inplace=True)

#Removing one of the pieces of data due to incomplete data
data_full_recording = data_full_recording[data_full_recording['testerKey'] != 'cohtoi']

#Swapping the binary flag of choice 6 to reflect choices 7 and 8 on which choice is the active one and which is the passive one
data.loc[data["passage"] == "CHOICE 6 - Pull the lever", "playerChoice"] = 1 - data.loc[data["passage"] == "CHOICE 6 - Pull the lever", "playerChoice"]

psg = 0
perc_diff = []
count = 0
#Taking the perceived difficulty of that specific passage and putting them all into one column so each row matches the passage at hand
for i, j in data.iterrows():
    if j.index[-1] == "passage_num":
        psg = j.iloc[-1]
        perc_diff.append(j[f"diff_{psg}"])

    if j.iloc[0] == "vwbjno":
        print("Hello")

data["perc_diff"] = perc_diff
data["perc_diff"] = data["perc_diff"].astype(int)

In [8]:
data_full_recording.to_csv("../Csv Files/MM_Full_Data.csv", index = False)
data.head()

,Recording,testerKey,TOI,Interval,AOI,Total_duration_of_fixations,Average_duration_of_fixations,Minimum_duration_of_fixations,Maximum_duration_of_fixations,Number_of_fixations,...,choiceOneMorality,choiceOnePosition,choiceTwo,choiceTwoMorality,choiceTwoPosition,prevMoralityValue,currentMoralityValue,swapped,passage_num,perc_diff
0,aagzig,aagzig,IM2 HeatMap,6,Left Button,699,17.0,16.0,17.0,42,...,15,Right,Do nothing.,-15,Left,30,45,False,6,4
1,aagzig,aagzig,IM2 HeatMap,6,Left Score,150,17.0,16.0,17.0,9,...,15,Right,Do nothing.,-15,Left,30,45,False,6,4
2,aagzig,aagzig,IM2 HeatMap,6,Morality Meter,0,NaN,NaN,NaN,0,...,15,Right,Do nothing.,-15,Left,30,45,False,6,4
3,aagzig,aagzig,IM2 HeatMap,6,Right Button,1048,17.0,16.0,17.0,63,...,15,Right,Do nothing.,-15,Left,30,45,False,6,4
4,aagzig,aagzig,IM2 HeatMap,6,Right Score,283,17.0,16.0,17.0,17,...,15,Right,Do nothing.,-15,Left,30,45,False,6,4


In [9]:
#AOI features we are including
aoi_info = ["Total_duration_of_fixations", "Average_duration_of_fixations","Average_pupil_diameter", "Number_of_fixations", "Time_to_first_fixation"]

#Columns we would like to include from the dataframe and how we want them to be in the new dataframe
columns = ["testerKey", "Interval", "Gender", "Age", "moralScore", "swapped", "playerChoice"] + \
          [f"{feature}_{aoi}" for aoi in ["Lb","LbScore", "Rb","RbScore", "text", "MM"] for feature in aoi_info] + \
          ["perc_diff", "responseTime", "family_Score", "loyalty_Score", "recip_Score", "bravery_Score", "respect_Score", "fairness_Score", "property_Score"]

#Creating an empty dataframe with the columns mentioned
updated_data = pd.DataFrame(columns=columns)

grouped = data.groupby(["testerKey", "Interval"])
rows = []

#Filling the rows of the new dataframe by grouping the rows and taking the relevant information
for (testerKey), group in grouped:
    row = {
        'testerKey': group["testerKey"].iloc[0],
        'Interval': group["Interval"].iloc[0],
        'moralScore': group["currentMoralityValue"].iloc[0] - group["prevMoralityValue"].iloc[0],
        'swapped': group["swapped"].iloc[0],
        "playerChoice": group["playerChoice"].iloc[0],
        'Gender': group["Gender"].iloc[0],
        'Age': group["Age"].iloc[0],
        'perc_diff': group["perc_diff"].iloc[0],
        'responseTime': group["responseTime"].iloc[0],
        'choiceOnePosition': group["choiceOnePosition"].iloc[0],
        #Taking the average of the MAC Survey statements to have an average category score
        "family_Score": (group["RF1"].iloc[0]+group["RF2"].iloc[0]+group["RF3"].iloc[0]+group["JF1"].iloc[0]+group["JF2"].iloc[0]+group["JF3"].iloc[0])/6, 
        "loyalty_Score":(group["RG1"].iloc[0]+group["RG2"].iloc[0]+group["RG3"].iloc[0]+group["JG1"].iloc[0]+group["JG2"].iloc[0]+group["JG3"].iloc[0])/6, 
        "recip_Score":(group["RR1"].iloc[0]+group["RR2"].iloc[0]+group["RR3"].iloc[0]+group["JR1"].iloc[0]+group["JR2"].iloc[0]+group["JR3"].iloc[0])/6, 
        "bravery_Score":(group["RH1"].iloc[0]+group["RH2"].iloc[0]+group["RH3"].iloc[0]+group["JH1"].iloc[0]+group["JH2"].iloc[0]+group["JH3"].iloc[0])/6, 
        "respect_Score":(group["RD1"].iloc[0]+group["RD2"].iloc[0]+group["RD3"].iloc[0]+group["JD1"].iloc[0]+group["JD2"].iloc[0]+group["JD3"].iloc[0])/6, 
        "fairness_Score":(group["RFa1"].iloc[0]+group["RFa2"].iloc[0]+group["RFa3"].iloc[0]+group["JFa1"].iloc[0]+group["JFa2"].iloc[0]+group["JFa3"].iloc[0])/6, 
        "property_Score":(group["RP1"].iloc[0]+group["RP2"].iloc[0]+group["RP3"].iloc[0]+group["JP1"].iloc[0]+group["JP2"].iloc[0]+group["JP3"].iloc[0])/6
    }

    #Creating the AOI dependant columns
    for _,aoi_row in group.iterrows():
        aoi = ""
        if 'Left Button' in aoi_row["AOI"]:
            aoi = 'Lb'
        elif 'Left Score' in aoi_row["AOI"]:
            aoi = 'Lb_Score'
        elif 'Right Button' in aoi_row["AOI"]:
            aoi = 'Rb'
        elif 'Right Score' in aoi_row["AOI"]:
            aoi = 'Rb_Score'
        elif 'Morality Meter' in aoi_row["AOI"]:
            aoi = 'MM'
        else:
            aoi = 'text'
        for feat in aoi_info:
            if aoi != "MM":
                row[f"{feat}_{aoi}"] = aoi_row[feat]
            else:
                temp = data_full_recording[data_full_recording["testerKey"] == group["testerKey"].iloc[0]]
                row[f"{feat}_{aoi}"] = temp[feat].values[0]

    rows.append(row)

#Filling the dataframe with the finished rows
updated_data = pd.DataFrame(rows)
updated_data.head()

,testerKey,Interval,moralScore,swapped,playerChoice,Gender,Age,perc_diff,responseTime,choiceOnePosition,...,Total_duration_of_fixations_Rb_Score,Average_duration_of_fixations_Rb_Score,Average_pupil_diameter_Rb_Score,Number_of_fixations_Rb_Score,Time_to_first_fixation_Rb_Score,Total_duration_of_fixations_text,Average_duration_of_fixations_text,Average_pupil_diameter_text,Number_of_fixations_text,Time_to_first_fixation_text
0,aagzig,6,15,False,1,Male,20,4,8.386885,Right,...,283,17.0,3.25397,17,5840.0,5958,17.0,3.38183,358,0.0
1,aagzig,7,-15,True,0,Male,20,5,11.077764,Left,...,465,17.0,3.16178,28,4764.0,4271,17.0,3.41623,258,0.0
2,aagzig,8,15,False,0,Male,20,1,18.660618,Right,...,0,NaN,NaN,0,NaN,15318,17.0,3.14845,921,0.0
3,ajtbsu,6,-15,True,1,Female,28,3,30.382484,Right,...,350,17.0,2.26395,21,5896.0,21903,17.0,2.53447,1315,0.0
4,ajtbsu,7,15,True,1,Female,28,4,37.033604,Right,...,233,17.0,2.77469,14,4735.0,30732,17.0,2.60739,1846,71.0


In [10]:
updated_data["condition"] = None
dir = "../DecisionData/converted"

#Adding one column in the new dataframe which contains an integer that represents the participants' group condition
for i in updated_data.index:
    for folder in os.listdir(dir):
        for key in os.listdir(os.path.join(dir, folder)):
            if key == updated_data["testerKey"].iloc[i]:
                if folder == "CM1":
                    updated_data.loc[i,"condition"] = 0
                elif folder == "CM2":
                    updated_data.loc[i,"condition"] = 1
                elif folder == "IM1":
                    updated_data.loc[i,"condition"] = 2
                else:
                    updated_data.loc[i,"condition"] = 3
                break

#Saving the final csv file to be used in our methods
updated_data.to_csv("../Csv Files/F_Merged.csv", index = False)